In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [2]:
combined_df = pd.read_parquet('.\\Output\\great_race_all_stages.parquet')

In [3]:
leg_cols = [c for c in combined_df.columns if c.startswith("LEG")]

# Pivot longer by leg
long_df = combined_df.melt(
    id_vars=['RANK', 'CAR', 'YEAR','FACTOR', 'ScYR', 'DIV', 'CREW', 'Stage'],
    value_vars=leg_cols,
    var_name='Leg',
    value_name='Time'
)

long_df['Early'] = long_df['Time'].apply(lambda x: '*' in str(x) if pd.notna(x) else False)
long_df['Discarded'] = long_df['Time'].apply(lambda x: '#' in str(x) if pd.notna(x) else False)
long_df['Time'] = long_df['Time'].str.replace(r"[\*\#]", "", regex=True)
# long_df['Time'] = pd.to_numeric(long_df['Time'], errors='coerce')

long_df['Leg'] = long_df['Leg'].str.extract(r'(\d+)').astype(int)
long_df = long_df.dropna(subset=['Time'])

In [4]:
# Convert "0m03s" or "0m03.25s" to total seconds
def m_s_to_seconds(val):
    try:
        val = str(val).strip()
        if "m" in val and "s" in val:
            minutes, rest = val.split("m")
            seconds = rest.replace("s", "")
            return int(minutes) * 60 + float(seconds)
        else:
            return float(val)  # fallback if numeric
    except:
        return None

# Select relevant columns
time_cols = [c for c in combined_df.columns if c.startswith(("LEG", "TOTAL", "PENALTY", "SCORE"))]

# Apply to all relevant columns
for col in time_cols:
    combined_df[col] = combined_df[col].apply(m_s_to_seconds)

time_cols = [c for c in long_df.columns if c.startswith(("Time"))]

# Apply to all relevant columns
for col in time_cols:
    long_df[col] = long_df[col].apply(m_s_to_seconds)

long_df['Actual_Time'] = long_df.apply(lambda row: row['Time'] if not row['Early'] else -row['Time'], axis=1)

long_df.to_parquet(".\\Output\\long_format_times.parquet", index=False)
long_df.to_excel(".\\Output\\long_format_times.xlsx", index=False)


In [5]:
# 1️⃣ Create counts without modifying long_df
plot_df = long_df.groupby(['Stage', 'Leg', 'Actual_Time'])['CAR'].count().reset_index()
plot_df = plot_df.rename(columns={'CAR': 'Count'})

# 2️⃣ Create a dataframe of Ownby/Wallace times
Ownby_Times_df = long_df[long_df['CREW'].str.contains("Ownby/Wallace", case=False)][['Stage', 'Leg', 'Actual_Time']].drop_duplicates()
Winner_Times_df = long_df[long_df['CREW'].str.contains("J & E Fredette", case=False)][['Stage', 'Leg', 'Actual_Time']].drop_duplicates()

# 3️⃣ Add a boolean column to indicate if the row matches Ownby's times
plot_df['IsOwnby'] = plot_df.apply(
    lambda row: ((row['Stage'], row['Leg'], row['Actual_Time']) 
                 in set(zip(Ownby_Times_df['Stage'], Ownby_Times_df['Leg'], Ownby_Times_df['Actual_Time']))),
    axis=1
)

plot_df['IsWinner'] = plot_df.apply(
    lambda row: ((row['Stage'], row['Leg'], row['Actual_Time']) 
                 in set(zip(Winner_Times_df['Stage'], Winner_Times_df['Leg'], Winner_Times_df['Actual_Time']))),
    axis=1
)

# 4️⃣ Create a color column for Plotly
plot_df['Color'] = 'blue'  # default color for other crews
plot_df.loc[plot_df['IsOwnby'], 'Color'] = 'red'
plot_df.loc[plot_df['IsWinner'], 'Color'] = 'green'

In [6]:
# 1️⃣ Create a categorical column for legend
plot_df['CrewType'] = 'Other'                   # default
plot_df.loc[plot_df['IsOwnby'], 'CrewType'] = 'Ownby'
plot_df.loc[plot_df['IsWinner'], 'CrewType'] = 'Winner'

# 2️⃣ Plot with facets and proper legend
fig = px.scatter(
    plot_df,
    x='Actual_Time',               # Time on x-axis
    y='Count',              # Count on y-axis
    color='CrewType',       # Ownby/Winner/Other for legend
    facet_row='Stage',      # columns = Stages
    facet_col='Leg',        # rows = Legs
    hover_data=['Count', 'CrewType'],
    title="Time distribution per LEG per Stage (Ownby/Wallace highlighted)",
    color_discrete_map={'Ownby':'red', 'Winner':'green', 'Other':'blue'}  # consistent colors
)

fig.update_layout(
    xaxis_title="Actual Time (seconds)",
    yaxis_title="Count",
    legend_title="Crew",
    height=400 + 100*plot_df['Leg'].nunique(),  # adjust height for number of leg rows
    width=1200,
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.show()

In [7]:
# 1️⃣ Create a categorical column for legend
plot_df['CrewType'] = 'Other'                   # default
plot_df.loc[plot_df['IsOwnby'], 'CrewType'] = 'Ownby'
plot_df.loc[plot_df['IsWinner'], 'CrewType'] = 'Winner'

# 2️⃣ Plot with facets and proper legend
fig = px.scatter(
    plot_df,
    x='Actual_Time',               # Time on x-axis
    y='Count',              # Count on y-axis
    color='CrewType',       # Ownby/Winner/Other for legend
    facet_col='Stage',      # columns = Stages
    # facet_row='Leg',        # rows = Legs
    hover_data=['Count', 'CrewType'],
    title="Time distribution per Stage (Ownby/Wallace highlighted)",
    color_discrete_map={'Ownby':'red', 'Winner':'green', 'Other':'blue'}  # consistent colors
)

fig.update_layout(
    xaxis_title="Time (seconds)",
    yaxis_title="Count",
    legend_title="Crew",
    # height=400 + 100*plot_df['Leg'].nunique(),  # adjust height for number of leg rows
    # width=1200,
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.show()

In [8]:

# 1️⃣ Extract Ownby times per Stage/Leg

# ownby_times = ownby_times.rename(columns={'Time':'OwnbyTime'})

# 2️⃣ Extract Winner times per Stage/Leg

# winner_times = winner_times.rename(columns={'Time':'WinnerTime'})

# 3️⃣ Merge Ownby and Winner times on Stage and Leg
paired_times = pd.merge(Ownby_Times_df, Winner_Times_df, on=['Stage','Leg'], how='inner', suffixes=('_Ownby', '_Winner'))

# 4️⃣ Plot scatter plot: Ownby on x, Winner on y
fig = px.scatter(
    paired_times,
    x='Actual_Time_Ownby',               # Ownby time on x-axis
    y='Actual_Time_Winner',               # Winner time on y-axis
    facet_col = 'Stage',
    # facet_row = 'Stage',
    # text='Stage',   # optional: label points with Stage
    hover_data=['Leg'],
    title="Ownby vs Winner Times per Stage and LEG",
    color_discrete_sequence=['purple']
)

# # 5️⃣ Add 45-degree reference line (equal times)
# fig.add_shape(
#     type="line",
#     x0=paired_times['Time_Ownby'].min(),
#     y0=paired_times['Time_Winner'].min(),
#     x1=paired_times['Time_Ownby'].max(),
#     y1=paired_times['Time_Winner'].max(),
#     line=dict(color="black", dash="dash"),
#     xref='x', yref='y'
# )

# Make axes equal scale
fig.update_layout(
    xaxis_title="Ownby Time (seconds)",
    yaxis_title="Winner Time (seconds)",
    # height=400 + 100*paired_times['Stage'].nunique(),  # adjust height for number of leg rows
    width=1200,
    margin=dict(l=50, r=50, t=80, b=50)
)
fig.show()

In [9]:
paired_times.groupby('Leg')[['Time_Ownby', 'Time_Winner']].sum()

KeyError: "Columns not found: 'Time_Winner', 'Time_Ownby'"